# 활성화 함수: GELU, SwiGLU 등 - 현대 활성화 함수 비교

- Tutorial ID: `adv-8-1`
- Tutorial: 활성화 함수: GELU, SwiGLU 등
- Section ID: `adv-8-1-1`
- Section: 현대 활성화 함수 비교


In [ ]:
# ============================================================
# 코드 읽는 법 — 현대 활성화 함수 비교
#
# 이 코드는 “정답을 한 번 실행”하는 용도가 아니라,
# 수학/아키텍처 개념이 실제 배열·텐서 연산으로 바뀌는 과정을
# 한 줄씩 추적하기 위한 실험 노트입니다.
#
# 학습 목표:
#   1) 입력 데이터가 어떤 중간 변수들을 거쳐 최종 출력으로 변환되는지 shape 중심으로 추적
#
# 읽는 순서:
#   1) 차원/하이퍼파라미터(batch_size, seq_len, d_model 등)를 먼저 확인합니다.
#   2) 입력 배열 X 또는 토큰/문서 데이터가 어떻게 만들어지는지 봅니다.
#   3) W_Q/W_K/W_V/W_O 같은 가중치 행렬이 어떤 공간으로 투영하는지 확인합니다.
#   4) @, matmul, softmax, mask, loss 등 핵심 연산 직후의 shape와 값을 출력으로 검증합니다.
#   5) seed, 차원, temperature, rank, top_k, expert 수 등을 바꿔 결과가 어떻게 변하는지 실험합니다.
#
# 주의:
#   - 숫자 하나하나를 외우기보다 “shape 변화”와 “정보가 이동하는 방향”을 보세요.
#   - torch/transformers/openai/vLLM 의존 코드는 Colab/로컬/서버 노트북 실행을 권장합니다.

In [ ]:
import numpy as np

print("=" * 60)
print("활성화 함수 비교")
print("=" * 60)

def relu(x):
    return np.maximum(0, x)

def gelu(x):
    return 0.5 * x * (1 + np.tanh(np.sqrt(2/np.pi) * (x + 0.044715 * x**3)))

def silu(x):
    return x * (1 / (1 + np.exp(-x)))

def sigmoid(x):
    return 1 / (1 + np.exp(-np.clip(x, -500, 500)))

# 값 비교
x_vals = np.array([-2, -1, -0.5, 0, 0.5, 1, 2])
print(f"\n{'x':>6} {'ReLU':>8} {'GELU':>8} {'SiLU':>8}")
print("-" * 35)
for x in x_vals:
    print(f"{x:>6.1f} {relu(x):>8.4f} {gelu(x):>8.4f} {silu(x):>8.4f}")

print("\nReLU: x<0에서 0 (죽은 뉴런)")
print("GELU/SiLU: x<0에서도 작은 값 (학습 지속)")

# SwiGLU
print("\nSwiGLU (LLaMA 스타일):")
d = 8
d_ff = 16
np.random.seed(42)
W_gate = np.random.randn(d, d_ff) * 0.1
W_value = np.random.randn(d, d_ff) * 0.1
W_out = np.random.randn(d_ff, d) * 0.1

x = np.random.randn(d)
gate = silu(x @ W_gate)
value = x @ W_value
out = (gate * value) @ W_out

print(f"  입력 dim: {d}")
print(f"  내부 dim: {d_ff}")
print(f"  출력 dim: {out.shape[0]}")
print(f"  활성화된 원소 비율: {np.mean(np.abs(gate) > 0.1):.1%}")
print("  → 게이트가 정보 흐름을 선택적으로 제어!")

# 희소 활성화 비교
print("\n희소 활성화 분석 (1000차원):")
X = np.random.randn(100, 1000)
for name, fn in [("ReLU", relu), ("GELU", gelu), ("SiLU", silu)]:
    acts = fn(X)
    zero_pct = np.mean(np.abs(acts) < 0.01)
    print(f"  {name}: {zero_pct:.1%} 비활성화")